# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an end-to-end template for loading and exploring the FAIR² dataset using the `mlcroissant` library. We follow the Croissant schema to access metadata, data, and key fields, referencing all record sets and columns by their `@id` fields as recommended.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Loaded Dataset Metadata:\n")
print(f"Name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Date Published: {metadata.datePublished}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Get all record set IDs and their details
record_sets = list(dataset.list_record_sets())

print(f"Found {len(record_sets)} record sets (tables):\n")
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '')}")
    if 'field' in rs:
        print(f"  Fields/Columns:")
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"    - @id: {field.get('@id', 'n/a')}, name: {field.get('name', '')}")
            else:
                print(f"    - @id: {field}")
    print()

# For exploration, print a preview (first few records) for each record set by @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Sample record from record set @id={rs_id}:")
    for record in dataset.records(record_set=rs_id):
        print(record)
        break  # just show one record for preview
    print()

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Gather the record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading data for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only store if not empty
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  -> Loaded {len(dataframes[rs_id])} records\n")
    else:
        print(f"  -> No records found\n")

# List all loaded tables
print("Loaded tables and their columns:")
for rs_id, df in dataframes.items():
    print(f"  Record set @id: {rs_id}")
    print(f"    Columns: {df.columns.tolist()}")
    print(f"    Sample rows:\n{df.head(2)}\n")

# Pick the main record set for further analysis (if only one, use that; otherwise, use the main clinical table or first table)
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Selected main record set for EDA: {main_rs_id}")
    print(f"DataFrame shape: {dataframes[main_rs_id].shape}")
    df = dataframes[main_rs_id]
else:
    raise ValueError("No dataframes loaded!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All columns are referenced by their `@id`s.

We will:
- Select a numeric field (e.g., 'age') by its column `@id`.
- Filter records with `age > 60`.
- Normalize the `age` column.
- Group data by a categorical field (e.g., 'sex') by its column `@id`.

> **Tip**: Replace the `age` and `sex` `@id` variables below with the actual `@id` values printed during the record set overview.

In [ ]:
# Example: Replace with the real @id values for your dataset
AGE_COL_ID = None
SEX_COL_ID = None
for col in df.columns:
    if "age" in col.lower():
        AGE_COL_ID = col
    if "sex" in col.lower() or "gender" in col.lower():
        SEX_COL_ID = col
print(f"Using age column @id: {AGE_COL_ID}")
print(f"Using sex column @id: {SEX_COL_ID}\n")

if AGE_COL_ID is not None:
    # Convert to numeric (in case it's not already)
    df[AGE_COL_ID] = pd.to_numeric(df[AGE_COL_ID], errors='coerce')
    
    # Filter age > 60
    threshold = 60
    filtered_df = df[df[AGE_COL_ID] > threshold].copy()
    print(f"Filtered records with {AGE_COL_ID} > {threshold}: {len(filtered_df)} records\n")
    print(filtered_df[[AGE_COL_ID, SEX_COL_ID]].head())
    
    # Normalize age
    col_norm = f"{AGE_COL_ID}_normalized"
    filtered_df[col_norm] = (filtered_df[AGE_COL_ID] - filtered_df[AGE_COL_ID].mean()) / filtered_df[AGE_COL_ID].std()
    print(f"\nNormalized {AGE_COL_ID} for filtered records:")
    print(filtered_df[[AGE_COL_ID, col_norm]].head())

    # Group by sex if available
    if SEX_COL_ID in filtered_df.columns:
        grouped_df = filtered_df.groupby(SEX_COL_ID)[AGE_COL_ID].agg(['count', 'mean', 'std'])
        print(f"\nGrouped (mean/std) by {SEX_COL_ID}:")
        print(grouped_df)
else:
    print("No suitable numeric field called 'age' was found for analysis.")

## 5. Visualization
Visualize age distributions and age by sex, using `@id` fields for columns when referencing data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if AGE_COL_ID is not None and not filtered_df.empty:
    # Histogram of age
    plt.figure(figsize=(8, 5))
    sns.histplot(df[AGE_COL_ID].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {AGE_COL_ID} (all records)")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.show()

    # Age by sex boxplot
    if SEX_COL_ID is not None:
        plt.figure(figsize=(7, 5))
        sns.boxplot(x=df[SEX_COL_ID], y=df[AGE_COL_ID])
        plt.title(f"{AGE_COL_ID} by {SEX_COL_ID}")
        plt.ylabel("Age")
        plt.xlabel(str(SEX_COL_ID))
        plt.show()
else:
    print("Visualization skipped: required fields not found or not enough data.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to load a dataset defined by the FAIR² Croissant schema, explored its metadata, and loaded clinical data tables by their `@id` references.

- The main record set and column names were discovered by schema introspection, and all subsequent analysis referenced these `@id`s.
- We demonstrated data extraction, EDA (filtering, normalization, grouping), and visualization using standard Python tools.
- This workflow serves as a reproducible, standards-compliant template for exploring Croissant datasets in a programmatic, scalable fashion.

You can adapt this notebook for other Croissant-based datasets simply by changing the dataset URL and referencing the correct `@id` keys for record sets and columns.